
# Rest-frame spectrum with stellar population ages


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")


ssp = tengri.load_ssp()

# --- Setup ---
REDSHIFT = 0.1
WAVE_REST = jnp.linspace(3500.0, 9200.0, 1000)

obs = tengri.Observation(
    spectroscopy=tengri.Spectroscopy(wave_obs=WAVE_REST * (1 + REDSHIFT)),
)

model = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={"type": "tsnorm", "all_params": tengri.FIXED},
    dust={
        "type": "two_component",
        "all_params": tengri.FIXED,
        "tau_bc": 0.2,
        "tau_diff": 0.1,
        "slope": -0.7,
    },
    redshift=tengri.Fixed(REDSHIFT),
)

# --- Baseline SFH parameters (same for both models) ---
baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))
baseline.update(
    sfh_tsnorm_skew=0.2,
    sfh_tsnorm_trunc=10.0,
    met_logzsol=0.0,
)

# --- Young and old stellar populations ---
young = {**baseline, "sfh_tsnorm_peak_lbt_gyr": 0.1, "sfh_tsnorm_width_gyr": 0.3}
old = {**baseline, "sfh_tsnorm_peak_lbt_gyr": 5.0, "sfh_tsnorm_width_gyr": 2.0}

young_spectrum = model.predict_spectrum(young, WAVE_REST * (1 + REDSHIFT))
old_spectrum = model.predict_spectrum(old, WAVE_REST * (1 + REDSHIFT))

# --- Plot ---
wave_rest_array = np.asarray(WAVE_REST)
fig, ax = plt.subplots(figsize=(10, 5))

ax.loglog(
    wave_rest_array,
    np.asarray(young_spectrum) / np.median(young_spectrum),
    color="C1",
    lw=1.5,
    label=r"Young (Age $\sim$ 0.1 Gyr)",
    alpha=0.8,
)
ax.loglog(
    wave_rest_array,
    np.asarray(old_spectrum) / np.median(old_spectrum),
    color="C0",
    lw=1.5,
    label=r"Old (Age $\sim$ 5 Gyr)",
    alpha=0.8,
)

# Feature annotations
ax.axvline(4102, color="gray", ls=":", lw=0.8, alpha=0.5)
ax.text(4102, 2.0, r"H$\delta$", fontsize=9, ha="center", color="gray")
ax.axvline(6563, color="gray", ls=":", lw=0.8, alpha=0.5)
ax.text(6563, 2.0, r"H$\alpha$", fontsize=9, ha="center", color="gray")

ax.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]")
ax.set_ylabel(r"Normalized $f_\lambda$")
ax.set_xlim(3500, 9200)
ax.set_ylim(0.2, 5.0)
ax.legend(frameon=False, loc="upper left", fontsize=10)

fig.tight_layout()
plt.savefig("plot_spectrum_fit.png", dpi=150, bbox_inches="tight")